# 01 Data Loading and Cleaning

**Project:** Sudan Civil War and Civilian Targeting Using ACLED  
**Course:** POLI3148 — Data Science in Politics and Public Administration  
**Notebook role:** Load the raw ACLED Sudan data, validate the download, clean reusable variables, and save only the files needed for later notebooks.

## Research focus

> **Where and why has violence against civilians intensified in Sudan's civil war?**  
> A geospatial and machine-learning analysis using ACLED event data, 2023–2025.

## What this notebook saves

To avoid unnecessary intermediate files, this notebook writes only two reusable processed datasets:

1. `data/processed/acled_sudan_events_clean.csv`  
   Cleaned event-level ACLED file for maps, actor analysis, event-type analysis, and network analysis.

2. `data/processed/acled_sudan_admin1_month_panel.csv`  
   Admin1-month panel for exploratory analysis and later machine-learning work.

Summary tables, missingness tables, and checks are displayed inside the notebook only. They are **not** saved as separate CSV files because they can be regenerated from the two processed datasets.

## 1. Setup

The paths below work in the recommended project structure:

```text
project_folder/
├── README.md
├── data/
│   ├── raw/
│   │   └── acled_sudan_2023_2025.csv
│   └── processed/
├── code/
│   └── 01_data_loading_cleaning.ipynb
├── docs/
│   └── index.html
└── note_on_ai_use.md
```

If the raw CSV is not in `data/raw/`, the notebook also searches `data/`, the project root, and the ChatGPT sandbox folder.

In [15]:
from pathlib import Path
from datetime import datetime
import re
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)

warnings.filterwarnings("ignore", category=FutureWarning)

In [16]:
# Detect project root.
# If this notebook is inside project_folder/code/, the project root is the parent folder.
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "code":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
CODE_DIR = PROJECT_ROOT / "code"
DOCS_DIR = PROJECT_ROOT / "docs"

for folder in [DATA_DIR, RAW_DIR, PROCESSED_DIR, CODE_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data folder: {RAW_DIR}")
print(f"Processed output folder: {PROCESSED_DIR}")

Project root: c:\Users\USER\Desktop\asm1
Raw data folder: c:\Users\USER\Desktop\asm1\data\raw
Processed output folder: c:\Users\USER\Desktop\asm1\data\processed


In [17]:
RAW_DATA_PATH = DATA_DIR / "raw" / "acled_sudan_2023_2025.csv"
print(f"Using raw file: {RAW_DATA_PATH}")

Using raw file: c:\Users\USER\Desktop\asm1\data\raw\acled_sudan_2023_2025.csv


## 2. Document the data and retrieval process

### 2.1 ACLED data-generating process

ACLED is an event-level dataset. Each row represents a reported political-violence, demonstration, or strategic-development event. ACLED records fields such as the event date, event type, sub-event type, actors, associated actors, administrative location, coordinates, fatalities, sources, and notes.

Important interpretation points for this project:

- ACLED records **reported events**, so the data reflect reporting availability and ACLED coding rules.
- Fatalities are estimates and should be interpreted carefully.
- `event_type` and `sub_event_type` classify the kind of event.
- `actor1`, `actor2`, `assoc_actor_1`, and `assoc_actor_2` support actor-level and optional network analysis.
- `latitude`, `longitude`, `admin1`, and `admin2` support spatial analysis.

### 2.2 My data-retrieval process

For this project, the raw dataset was downloaded from ACLED with these filters:

- **Country:** Sudan
- **Date range:** 2023-04-15 to 2025-04-22
- **Event types:** All ACLED event types
- **Purpose:** Analyze how the Sudan civil war has evolved geographically, with special attention to violence against civilians and actor patterns.

If you later re-download a newer file, update the date range above and re-run this notebook from top to bottom.

## 3. Load the raw ACLED file

In [18]:
df_raw = pd.read_csv(RAW_DATA_PATH, encoding="utf-8-sig", low_memory=False)

print(f"Raw rows: {df_raw.shape[0]:,}")
print(f"Raw columns: {df_raw.shape[1]:,}")
display(df_raw.head())

Raw rows: 16,004
Raw columns: 31


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,actor2,assoc_actor_2,inter2,interaction,civilian_targeting,iso,region,country,admin1,admin2,admin3,location,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp
0,SUD19670,2023-04-15,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),NaN,Identity militia,Civilians (Sudan),Government of Sudan (2019-),Civilians,Identity militia-Civilians,NaN,729,Northern Africa,Sudan,South Darfur,Nyala Janoub,NaN,Nyala,12.0556,24.8906,2,Darfur 24,Subnational,"Looting: On 15 April 2023, an unidentified arm...",0,NaN,1754958639
1,SUD19671,2023-04-16,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),NaN,Identity militia,Civilians (Sudan),Labor Group (Sudan),Civilians,Identity militia-Civilians,NaN,729,Northern Africa,Sudan,Central Darfur,Zalingi,NaN,Zalingei,12.9075,23.4706,1,Darfur 24; Radio Dabanga,Subnational-National,"Looting: On 16 April 2023, an unidentified arm...",0,NaN,1753971757
2,SUD19672,2023-04-19,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),NaN,Identity militia,Civilians (Sudan),MSF: Doctors Without Borders,Civilians,Identity militia-Civilians,NaN,729,Northern Africa,Sudan,South Darfur,Nyala Janoub,NaN,Nyala,12.0556,24.8906,1,Sudan Nile; Twitter,New media-National,"Looting: On 19 April 2023, armed men (coded as...",0,NaN,1754958639
3,SUD19674,2023-04-16,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),NaN,Identity militia,Civilians (Sudan),NaN,Civilians,Identity militia-Civilians,NaN,729,Northern Africa,Sudan,North Darfur,Kebkabiya,NaN,Kebkabiya,13.6469,24.0991,1,Radio Dabanga,National,"Looting: On 16 April 2023, an unidentified arm...",0,NaN,1753971757
4,SUD19822,2023-04-25,2023,1,Political violence,Battles,Armed clash,Unidentified Armed Group (Sudan),NaN,Political militia,Police Forces of Sudan (2019-) Prison Guards,NaN,State forces,State forces-Political militia,NaN,729,Northern Africa,Sudan,North Darfur,Al Fasher,NaN,El Fasher,13.6264,25.3559,1,Darfur 24,Subnational,"On 25 April 2023, an armed group in cars and o...",0,NaN,1753971757


## 4. Standardize column names and validate the download

This section checks whether the file has the expected ACLED fields and whether the country/date scope matches the intended Sudan download.

In [19]:
# Standardize column names in case future downloads use slightly different capitalization or spacing.
df_raw.columns = (
    df_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

expected_columns = {
    "event_id_cnty", "event_date", "year", "time_precision", "disorder_type",
    "event_type", "sub_event_type", "actor1", "assoc_actor_1", "inter1",
    "actor2", "assoc_actor_2", "inter2", "interaction", "civilian_targeting",
    "iso", "region", "country", "admin1", "admin2", "admin3", "location",
    "latitude", "longitude", "geo_precision", "source", "source_scale",
    "notes", "fatalities", "tags", "timestamp"
}

missing_expected = sorted(expected_columns - set(df_raw.columns))
extra_columns = sorted(set(df_raw.columns) - expected_columns)

print("Missing expected columns:", missing_expected if missing_expected else "None")
print("Extra columns:", extra_columns if extra_columns else "None")

critical_columns = [
    "event_id_cnty", "event_date", "event_type", "sub_event_type",
    "actor1", "actor2", "country", "admin1", "admin2",
    "latitude", "longitude", "fatalities", "civilian_targeting"
]

missing_critical = [col for col in critical_columns if col not in df_raw.columns]
if missing_critical:
    raise ValueError(f"Missing critical columns needed for the project: {missing_critical}")

Missing expected columns: None
Extra columns: None


In [20]:
# Validate the intended download scope.
DATE_START = pd.Timestamp("2023-04-15")
DATE_END = pd.Timestamp("2025-04-22")
TARGET_COUNTRY = "Sudan"

date_check = pd.to_datetime(df_raw["event_date"], errors="coerce")

validation_summary = pd.Series({
    "raw_rows": len(df_raw),
    "raw_columns": df_raw.shape[1],
    "min_event_date": date_check.min(),
    "max_event_date": date_check.max(),
    "countries": ", ".join(sorted(df_raw["country"].dropna().astype(str).unique())),
    "admin1_units": df_raw["admin1"].nunique(dropna=True),
    "event_types": df_raw["event_type"].nunique(dropna=True),
    "duplicate_event_ids": df_raw["event_id_cnty"].duplicated().sum(),
    "total_fatalities_raw": pd.to_numeric(df_raw["fatalities"], errors="coerce").sum()
})

display(validation_summary.to_frame("value"))

print("Event types in raw file:")
display(df_raw["event_type"].value_counts(dropna=False).to_frame("events"))

print("Date-range check:")
print(f"Expected start: {DATE_START.date()} | Observed start: {date_check.min().date()}")
print(f"Expected end:   {DATE_END.date()} | Observed end:   {date_check.max().date()}")

,value
raw_rows,16004
raw_columns,31
min_event_date,2023-04-15 00:00:00
max_event_date,2025-04-22 00:00:00
countries,Sudan
admin1_units,19
event_types,6
duplicate_event_ids,0
total_fatalities_raw,43256


Event types in raw file:


,events
event_type,
Battles,5112
Explosions/Remote violence,3830
Strategic developments,3419
Violence against civilians,3374
Protests,248
Riots,21


Date-range check:
Expected start: 2023-04-15 | Observed start: 2023-04-15
Expected end:   2025-04-22 | Observed end:   2025-04-22


## 5. Clean the event-level data

Cleaning choices:

1. Keep only Sudan events in the intended date range.
2. Preserve all ACLED event types.
3. Convert dates, coordinates, fatalities, precision fields, and timestamps to appropriate data types.
4. Strip unnecessary whitespace from text fields.
5. Drop duplicate `event_id_cnty` values if they appear.
6. Keep event-level records as the main reusable cleaned dataset.

In [21]:
df = df_raw.copy()

# Clean text fields.
text_columns = df.select_dtypes(include=["object"]).columns
for col in text_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA})
    )

# Convert dates.
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce").dt.normalize()

# Convert numeric fields.
numeric_columns = ["year", "time_precision", "inter1", "inter2", "interaction", "iso", "geo_precision", "timestamp"]
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["fatalities"] = pd.to_numeric(df["fatalities"], errors="coerce").fillna(0).astype(int)

# ACLED timestamp is a Unix timestamp in many exports. Keep a readable UTC conversion for transparency.
if "timestamp" in df.columns:
    df["acled_timestamp_utc"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce", utc=True)

# Keep only intended project scope.
before_scope_filter = len(df)
df = df.loc[
    (df["country"].eq(TARGET_COUNTRY)) &
    (df["event_date"].between(DATE_START, DATE_END, inclusive="both"))
].copy()
after_scope_filter = len(df)

# Drop rows without a date or event ID because those cannot be reliably analyzed.
before_required_filter = len(df)
df = df.dropna(subset=["event_date", "event_id_cnty"]).copy()
after_required_filter = len(df)

# Drop duplicate event IDs if present.
before_dedup = len(df)
df = df.drop_duplicates(subset=["event_id_cnty"], keep="first").copy()
after_dedup = len(df)

# Fatalities should not be negative. If a future file contains negative values, set them to zero.
df.loc[df["fatalities"] < 0, "fatalities"] = 0

print(f"Rows before country/date filter: {before_scope_filter:,}")
print(f"Rows after country/date filter:  {after_scope_filter:,}")
print(f"Rows removed for missing date/id: {before_required_filter - after_required_filter:,}")
print(f"Duplicate event IDs removed:      {before_dedup - after_dedup:,}")
print(f"Final cleaned event rows:         {len(df):,}")

Rows before country/date filter: 16,004
Rows after country/date filter:  16,004
Rows removed for missing date/id: 0
Duplicate event IDs removed:      0
Final cleaned event rows:         16,004


## 6. Create reusable analysis variables

These variables are designed for later notebooks:

- **Time-series analysis:** `month`, `year_month`, `quarter`, `days_since_war_start`
- **Civilian-targeting analysis:** `is_civilian_targeting`, `is_violence_against_civilians`
- **Actor analysis:** `saf_involved`, `rsf_involved`, `civilian_actor_involved`
- **Spatial analysis:** `admin1`, `admin2`, `latitude`, `longitude`, `macro_region`
- **Machine learning:** admin1-month panel variables created in the next section

In [22]:
# Time variables.
df["month"] = df["event_date"].dt.to_period("M").dt.to_timestamp()
df["year_month"] = df["event_date"].dt.strftime("%Y-%m")
df["quarter"] = df["event_date"].dt.to_period("Q").astype(str)
df["days_since_war_start"] = (df["event_date"] - DATE_START).dt.days
df["month_number"] = df["event_date"].dt.month
df["year_clean"] = df["event_date"].dt.year

# Event-type flags.
df["is_battle"] = df["event_type"].eq("Battles")
df["is_explosion_remote"] = df["event_type"].eq("Explosions/Remote violence")
df["is_strategic_development"] = df["event_type"].eq("Strategic developments")
df["is_violence_against_civilians"] = df["event_type"].eq("Violence against civilians")
df["is_protest_or_riot"] = df["event_type"].isin(["Protests", "Riots"])

# Civilian targeting flag.
df["is_civilian_targeting"] = (
    df["civilian_targeting"]
    .fillna("")
    .str.contains("civilian targeting", case=False, regex=False)
)

# Fatality variables for later visuals.
df["has_fatalities"] = df["fatalities"].gt(0)
df["fatality_category"] = pd.cut(
    df["fatalities"],
    bins=[-1, 0, 4, 24, 99, np.inf],
    labels=["0", "1-4", "5-24", "25-99", "100+"]
)

# Combine actor fields for transparent actor flags and optional search.
actor_columns = ["actor1", "assoc_actor_1", "actor2", "assoc_actor_2"]
for col in actor_columns:
    if col not in df.columns:
        df[col] = pd.NA

df["actor_text"] = (
    df[actor_columns]
    .fillna("")
    .agg("; ".join, axis=1)
    .str.replace(r"\s*;\s*;\s*", "; ", regex=True)
    .str.strip("; ")
)

def cell_mentions_rsf(value):
    if pd.isna(value):
        return False
    return "rapid support forces" in str(value).lower()

def cell_mentions_saf(value):
    # ACLED commonly codes the Sudanese state military as
    # 'Military Forces of Sudan (2019-)'.
    #
    # This function excludes the older/former RSF coding string
    # 'Military Forces of Sudan (2019-2023) Rapid Support Forces'
    # so RSF is not accidentally counted as SAF.
    if pd.isna(value):
        return False
    text = str(value).lower()
    mentions_state_military = "military forces of sudan (2019-)" in text
    mentions_rsf = "rapid support forces" in text
    return mentions_state_military and not mentions_rsf

df["rsf_involved"] = df[actor_columns].apply(lambda row: any(cell_mentions_rsf(x) for x in row), axis=1)
df["saf_involved"] = df[actor_columns].apply(lambda row: any(cell_mentions_saf(x) for x in row), axis=1)
df["civilian_actor_involved"] = df["actor_text"].str.contains("Civilian", case=False, na=False)

# Macro-regions for cleaner Sudan maps and narratives.
macro_region_map = {
    "Central Darfur": "Darfur",
    "East Darfur": "Darfur",
    "North Darfur": "Darfur",
    "South Darfur": "Darfur",
    "West Darfur": "Darfur",
    "North Kordofan": "Kordofan",
    "South Kordofan": "Kordofan",
    "West Kordofan": "Kordofan",
    "Khartoum": "Khartoum",
    "Gedaref": "Eastern Sudan",
    "Kassala": "Eastern Sudan",
    "Red Sea": "Eastern Sudan",
    "Al Jazirah": "Central/Nile",
    "Blue Nile": "Central/Nile",
    "Sennar": "Central/Nile",
    "White Nile": "Central/Nile",
    "Northern": "Northern/Nile",
    "River Nile": "Northern/Nile",
    "Abyei": "Abyei",
}
df["macro_region"] = df["admin1"].map(macro_region_map).fillna("Other/Unknown")

# Make boolean columns explicit integer flags for easier CSV reuse in later notebooks.
flag_columns = [
    "is_battle", "is_explosion_remote", "is_strategic_development",
    "is_violence_against_civilians", "is_protest_or_riot",
    "is_civilian_targeting", "has_fatalities", "rsf_involved",
    "saf_involved", "civilian_actor_involved"
]
for col in flag_columns:
    df[col] = df[col].astype(int)

display(df.head())

,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,actor2,assoc_actor_2,inter2,interaction,civilian_targeting,iso,region,country,admin1,admin2,admin3,location,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,acled_timestamp_utc,month,year_month,quarter,days_since_war_start,month_number,year_clean,is_battle,is_explosion_remote,is_strategic_development,is_violence_against_civilians,is_protest_or_riot,is_civilian_targeting,has_fatalities,fatality_category,actor_text,rsf_involved,saf_involved,civilian_actor_involved,macro_region
0,SUD19670,2023-04-15,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),<NA>,<NA>,Civilians (Sudan),Government of Sudan (2019-),<NA>,<NA>,<NA>,729,Northern Africa,Sudan,South Darfur,Nyala Janoub,NaN,Nyala,12.0556,24.8906,2,Darfur 24,Subnational,"Looting: On 15 April 2023, an unidentified arm...",0,<NA>,1754958639,2025-08-12 00:30:39+00:00,2023-04-01,2023-04,2023Q2,0,4,2023,0,0,1,0,0,0,0,0,Darfur Communal Militia (Sudan); Civilians (Su...,0,0,1,Darfur
1,SUD19671,2023-04-16,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),<NA>,<NA>,Civilians (Sudan),Labor Group (Sudan),<NA>,<NA>,<NA>,729,Northern Africa,Sudan,Central Darfur,Zalingi,NaN,Zalingei,12.9075,23.4706,1,Darfur 24; Radio Dabanga,Subnational-National,"Looting: On 16 April 2023, an unidentified arm...",0,<NA>,1753971757,2025-07-31 14:22:37+00:00,2023-04-01,2023-04,2023Q2,1,4,2023,0,0,1,0,0,0,0,0,Darfur Communal Militia (Sudan); Civilians (Su...,0,0,1,Darfur
2,SUD19672,2023-04-19,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),<NA>,<NA>,Civilians (Sudan),MSF: Doctors Without Borders,<NA>,<NA>,<NA>,729,Northern Africa,Sudan,South Darfur,Nyala Janoub,NaN,Nyala,12.0556,24.8906,1,Sudan Nile; Twitter,New media-National,"Looting: On 19 April 2023, armed men (coded as...",0,<NA>,1754958639,2025-08-12 00:30:39+00:00,2023-04-01,2023-04,2023Q2,4,4,2023,0,0,1,0,0,0,0,0,Darfur Communal Militia (Sudan); Civilians (Su...,0,0,1,Darfur
3,SUD19674,2023-04-16,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),<NA>,<NA>,Civilians (Sudan),<NA>,<NA>,<NA>,<NA>,729,Northern Africa,Sudan,North Darfur,Kebkabiya,NaN,Kebkabiya,13.6469,24.0991,1,Radio Dabanga,National,"Looting: On 16 April 2023, an unidentified arm...",0,<NA>,1753971757,2025-07-31 14:22:37+00:00,2023-04-01,2023-04,2023Q2,1,4,2023,0,0,1,0,0,0,0,0,Darfur Communal Militia (Sudan); Civilians (Su...,0,0,1,Darfur
4,SUD19822,2023-04-25,2023,1,Political violence,Battles,Armed clash,Unidentified Armed Group (Sudan),<NA>,<NA>,Police Forces of Sudan (2019-) Prison Guards,<NA>,<NA>,<NA>,<NA>,729,Northern Africa,Sudan,North Darfur,Al Fasher,NaN,El Fasher,13.6264,25.3559,1,Darfur 24,Subnational,"On 25 April 2023, an armed group in cars and o...",0,<NA>,1753971757,2025-07-31 14:22:37+00:00,2023-04-01,2023-04,2023Q2,10,4,2023,1,0,0,0,0,0,0,0,Unidentified Armed Group (Sudan); Police Force...,0,0,0,Darfur


## 7. Data-quality checks

These checks are shown inside the notebook only. They are not saved as separate processed files because later notebooks can regenerate them from `acled_sudan_events_clean.csv`.

In [23]:
quality_checks = pd.Series({
    "rows_cleaned": len(df),
    "unique_event_ids": df["event_id_cnty"].nunique(dropna=True),
    "duplicate_event_ids": df["event_id_cnty"].duplicated().sum(),
    "min_event_date": df["event_date"].min(),
    "max_event_date": df["event_date"].max(),
    "admin1_units": df["admin1"].nunique(dropna=True),
    "admin2_units": df["admin2"].nunique(dropna=True),
    "missing_event_date": df["event_date"].isna().sum(),
    "missing_admin1": df["admin1"].isna().sum(),
    "missing_admin2": df["admin2"].isna().sum(),
    "missing_latitude": df["latitude"].isna().sum(),
    "missing_longitude": df["longitude"].isna().sum(),
    "total_fatalities_cleaned": df["fatalities"].sum(),
    "civilian_targeting_events": df["is_civilian_targeting"].sum(),
    "violence_against_civilians_events": df["is_violence_against_civilians"].sum(),
    "saf_involved_events": df["saf_involved"].sum(),
    "rsf_involved_events": df["rsf_involved"].sum(),
})

display(quality_checks.to_frame("value"))

,value
rows_cleaned,16004
unique_event_ids,16004
duplicate_event_ids,0
min_event_date,2023-04-15 00:00:00
max_event_date,2025-04-22 00:00:00
admin1_units,19
admin2_units,172
missing_event_date,0
missing_admin1,0
missing_admin2,0


In [24]:
print("Missingness, top 20 fields:")
missingness = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_percent")
    .to_frame()
)

display(missingness.head(20))

print("Event type distribution:")
display(df["event_type"].value_counts(dropna=False).to_frame("events"))

print("Top 10 admin1 units by event count:")
display(df["admin1"].value_counts(dropna=False).head(10).to_frame("events"))

Missingness, top 20 fields:


,missing_percent
inter1,100.000000
inter2,100.000000
interaction,100.000000
admin3,100.000000
tags,97.075731
assoc_actor_1,88.546613
assoc_actor_2,80.598600
civilian_targeting,72.075731
actor2,13.215446
year_month,0.000000


Event type distribution:


,events
event_type,
Battles,5112
Explosions/Remote violence,3830
Strategic developments,3419
Violence against civilians,3374
Protests,248
Riots,21


Top 10 admin1 units by event count:


,events
admin1,
Khartoum,6838
Al Jazirah,2199
North Darfur,1874
South Darfur,731
North Kordofan,678
Sennar,483
South Kordofan,426
West Darfur,421
White Nile,414


In [25]:
# Coordinate sanity check.
# These broad bounds should cover Sudan and nearby border-coded events without being overly strict.
coordinate_issues = df.loc[
    df["latitude"].isna() |
    df["longitude"].isna() |
    ~df["latitude"].between(7, 24) |
    ~df["longitude"].between(20, 40),
    ["event_id_cnty", "event_date", "admin1", "admin2", "location", "latitude", "longitude", "geo_precision", "notes"]
]

print(f"Potential coordinate issues: {len(coordinate_issues):,}")
display(coordinate_issues.head(10))

Potential coordinate issues: 0


,event_id_cnty,event_date,admin1,admin2,location,latitude,longitude,geo_precision,notes


## 8. Create an admin1-month panel for EDA and machine learning

This is the only aggregated processed dataset saved by this cleaning notebook because it will be useful later.

Possible later research question:

> Can previous-month conflict patterns help identify which Sudanese states are at higher risk of civilian-targeting violence in the next month?

Each row is one Sudanese `admin1` state-month. Months with zero recorded events are included so time-series and ML models do not silently ignore quiet periods.

In [26]:
# Build a complete admin1-month panel so months with zero events are represented.
admin1_units = sorted(df["admin1"].dropna().unique())
all_months = pd.date_range(df["month"].min(), df["month"].max(), freq="MS")
panel_index = pd.MultiIndex.from_product([admin1_units, all_months], names=["admin1", "month"])

# Helper for counting distinct actors across actor1 and actor2.
actor_long = (
    df[["admin1", "month", "actor1", "actor2"]]
    .melt(id_vars=["admin1", "month"], value_vars=["actor1", "actor2"], value_name="actor")
    .dropna(subset=["actor"])
)

distinct_actor_counts = (
    actor_long
    .groupby(["admin1", "month"])["actor"]
    .nunique()
    .rename("distinct_actor_count")
)

admin_month_observed = (
    df.groupby(["admin1", "month"])
    .agg(
        events=("event_id_cnty", "count"),
        fatalities=("fatalities", "sum"),
        civilian_targeting_events=("is_civilian_targeting", "sum"),
        violence_against_civilians_events=("is_violence_against_civilians", "sum"),
        battles=("is_battle", "sum"),
        explosions_remote=("is_explosion_remote", "sum"),
        strategic_developments=("is_strategic_development", "sum"),
        protests_riots=("is_protest_or_riot", "sum"),
        saf_events=("saf_involved", "sum"),
        rsf_events=("rsf_involved", "sum"),
        civilian_actor_events=("civilian_actor_involved", "sum"),
        fatal_event_count=("has_fatalities", "sum"),
        unique_locations=("location", "nunique"),
        unique_admin2=("admin2", "nunique"),
        mean_latitude=("latitude", "mean"),
        mean_longitude=("longitude", "mean"),
    )
    .join(distinct_actor_counts, how="left")
    .reindex(panel_index)
    .reset_index()
)

count_columns = [
    "events", "fatalities", "civilian_targeting_events",
    "violence_against_civilians_events", "battles", "explosions_remote",
    "strategic_developments", "protests_riots", "saf_events", "rsf_events",
    "civilian_actor_events", "fatal_event_count", "unique_locations",
    "unique_admin2", "distinct_actor_count"
]

for col in count_columns:
    admin_month_observed[col] = admin_month_observed[col].fillna(0).astype(int)

# Keep macro region at the admin1 level.
admin1_macro = df[["admin1", "macro_region"]].drop_duplicates()
admin_month = admin_month_observed.merge(admin1_macro, on="admin1", how="left")

# Add time variables.
admin_month["year"] = admin_month["month"].dt.year
admin_month["month_number"] = admin_month["month"].dt.month
admin_month["year_month"] = admin_month["month"].dt.strftime("%Y-%m")
admin_month["months_since_war_start"] = (
    (admin_month["month"].dt.year - DATE_START.year) * 12
    + (admin_month["month"].dt.month - DATE_START.month)
)

# Lagged features to reduce leakage in later ML work.
lag_base_columns = [
    "events", "fatalities", "civilian_targeting_events",
    "violence_against_civilians_events", "battles", "explosions_remote",
    "strategic_developments", "protests_riots", "saf_events", "rsf_events",
    "civilian_actor_events", "fatal_event_count", "distinct_actor_count",
    "unique_locations", "unique_admin2"
]

admin_month = admin_month.sort_values(["admin1", "month"]).reset_index(drop=True)

for col in lag_base_columns:
    admin_month[f"{col}_lag1"] = (
        admin_month
        .groupby("admin1", group_keys=False)[col]
        .shift(1)
        .fillna(0)
        .astype(int)
    )

# Three-month rolling features based only on previous months.
for col in ["events", "fatalities", "civilian_targeting_events", "battles", "explosions_remote"]:
    admin_month[f"{col}_prev3mo"] = (
        admin_month
        .groupby("admin1", group_keys=False)[col]
        .apply(lambda s: s.shift(1).rolling(window=3, min_periods=1).sum())
        .fillna(0)
        .astype(int)
    )

# Next-month targets for later ML. The final month for each admin1 has no next-month target.
next_civ = admin_month.groupby("admin1")["civilian_targeting_events"].shift(-1)
admin_month["civilian_targeting_events_next_month"] = next_civ

admin_month["target_any_civilian_targeting_next_month"] = pd.NA
valid_next = next_civ.notna()
admin_month.loc[valid_next, "target_any_civilian_targeting_next_month"] = (next_civ.loc[valid_next] > 0).astype(int)
admin_month["target_any_civilian_targeting_next_month"] = admin_month["target_any_civilian_targeting_next_month"].astype("Int64")

high_threshold = admin_month.loc[valid_next, "civilian_targeting_events_next_month"].quantile(0.75)
admin_month["target_high_civilian_targeting_next_month"] = pd.NA
admin_month.loc[valid_next, "target_high_civilian_targeting_next_month"] = (
    admin_month.loc[valid_next, "civilian_targeting_events_next_month"] > high_threshold
).astype(int)
admin_month["target_high_civilian_targeting_next_month"] = admin_month["target_high_civilian_targeting_next_month"].astype("Int64")

# Reorder columns for readability.
front_columns = [
    "admin1", "macro_region", "month", "year_month", "year", "month_number",
    "months_since_war_start", "events", "fatalities",
    "civilian_targeting_events", "violence_against_civilians_events",
    "battles", "explosions_remote", "strategic_developments",
    "protests_riots", "saf_events", "rsf_events", "distinct_actor_count",
    "unique_locations", "unique_admin2",
    "civilian_targeting_events_next_month",
    "target_any_civilian_targeting_next_month",
    "target_high_civilian_targeting_next_month"
]
remaining_columns = [col for col in admin_month.columns if col not in front_columns]
admin_month = admin_month[front_columns + remaining_columns]

print(f"Admin1-month panel rows: {len(admin_month):,}")
print(f"Admin1 units: {admin_month['admin1'].nunique():,}")
print(f"Months: {admin_month['month'].nunique():,}")
print(f"High civilian-targeting threshold for next-month target: {high_threshold:.2f}")

display(admin_month.head(10))

Admin1-month panel rows: 475
Admin1 units: 19
Months: 25
High civilian-targeting threshold for next-month target: 8.00


,admin1,macro_region,month,year_month,year,month_number,months_since_war_start,events,fatalities,civilian_targeting_events,violence_against_civilians_events,battles,explosions_remote,strategic_developments,protests_riots,saf_events,rsf_events,distinct_actor_count,unique_locations,unique_admin2,civilian_targeting_events_next_month,target_any_civilian_targeting_next_month,target_high_civilian_targeting_next_month,civilian_actor_events,fatal_event_count,mean_latitude,mean_longitude,events_lag1,fatalities_lag1,civilian_targeting_events_lag1,violence_against_civilians_events_lag1,battles_lag1,explosions_remote_lag1,strategic_developments_lag1,protests_riots_lag1,saf_events_lag1,rsf_events_lag1,civilian_actor_events_lag1,fatal_event_count_lag1,distinct_actor_count_lag1,unique_locations_lag1,unique_admin2_lag1,events_prev3mo,fatalities_prev3mo,civilian_targeting_events_prev3mo,battles_prev3mo,explosions_remote_prev3mo
0,Abyei,Abyei,2023-04-01,2023-04,2023,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6.0,1,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Abyei,Abyei,2023-05-01,2023-05,2023,5,1,12,36,6,6,4,0,2,0,0,0,10,10,1,1.0,1,0,8,8,9.527508,28.551233,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Abyei,Abyei,2023-06-01,2023-06,2023,6,2,6,21,1,1,4,0,1,0,0,0,4,3,1,2.0,1,0,2,5,9.541467,28.515283,12,36,6,6,4,0,2,0,0,0,8,8,10,10,1,12,36,6,4,0
3,Abyei,Abyei,2023-07-01,2023-07,2023,7,3,2,0,2,2,0,0,0,0,0,0,4,1,1,1.0,1,0,2,0,9.589900,28.448600,6,21,1,1,4,0,1,0,0,0,2,5,4,3,1,18,57,7,8,0
4,Abyei,Abyei,2023-08-01,2023-08,2023,8,4,7,11,1,1,5,0,0,1,0,0,6,6,1,5.0,1,0,1,6,9.520714,28.536986,2,0,2,2,0,0,0,0,0,0,2,0,4,1,1,20,57,9,8,0
5,Abyei,Abyei,2023-09-01,2023-09,2023,9,5,7,32,5,5,2,0,0,0,0,0,5,4,1,1.0,1,0,5,6,9.507557,28.490900,7,11,1,1,5,0,0,1,0,0,1,6,6,6,1,15,32,4,9,0
6,Abyei,Abyei,2023-10-01,2023-10,2023,10,6,9,17,1,1,5,0,2,1,0,0,13,4,1,9.0,1,1,1,5,9.546411,28.454622,7,32,5,5,2,0,0,0,0,0,5,6,5,4,1,16,43,8,7,0
7,Abyei,Abyei,2023-11-01,2023-11,2023,11,7,11,61,9,9,0,0,0,2,0,0,4,8,1,3.0,1,0,9,8,9.488536,28.488464,9,17,1,1,5,0,2,1,0,0,1,5,13,4,1,23,60,7,12,0
8,Abyei,Abyei,2023-12-01,2023-12,2023,12,8,18,42,3,3,10,0,1,4,0,0,6,10,1,8.0,1,0,4,9,9.450867,28.518578,11,61,9,9,0,0,0,2,0,0,9,8,4,8,1,27,110,15,7,0
9,Abyei,Abyei,2024-01-01,2024-01,2024,1,9,22,95,8,8,8,0,4,2,0,0,10,14,1,13.0,1,1,11,14,9.507786,28.524823,18,42,3,3,10,0,1,4,0,0,4,9,6,10,1,38,120,13,15,0


## 9. Save only the reusable processed datasets

This notebook intentionally saves only:

- the cleaned event-level file
- the admin1-month panel

No summary CSVs, chart files, or separate cleaning logs are saved here.

In [27]:
clean_events_path = PROCESSED_DIR / "acled_sudan_events_clean.csv"
admin_month_path = PROCESSED_DIR / "acled_sudan_admin1_month_panel.csv"

df.to_csv(clean_events_path, index=False)
admin_month.to_csv(admin_month_path, index=False)

print("Saved processed files:")
print(f"- {clean_events_path}")
print(f"- {admin_month_path}")

Saved processed files:
- c:\Users\USER\Desktop\asm1\data\processed\acled_sudan_events_clean.csv
- c:\Users\USER\Desktop\asm1\data\processed\acled_sudan_admin1_month_panel.csv


## 10. Final checks for later notebooks

In [28]:
# Re-read saved files to confirm they were written successfully.
check_events = pd.read_csv(clean_events_path, nrows=5)
check_admin_month = pd.read_csv(admin_month_path, nrows=5)

print("Cleaned event file preview:")
display(check_events)

print("Admin1-month panel preview:")
display(check_admin_month)

print("Notebook completed successfully.")

Cleaned event file preview:


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,actor2,assoc_actor_2,inter2,interaction,civilian_targeting,iso,region,country,admin1,admin2,admin3,location,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,acled_timestamp_utc,month,year_month,quarter,days_since_war_start,month_number,year_clean,is_battle,is_explosion_remote,is_strategic_development,is_violence_against_civilians,is_protest_or_riot,is_civilian_targeting,has_fatalities,fatality_category,actor_text,rsf_involved,saf_involved,civilian_actor_involved,macro_region
0,SUD19670,2023-04-15,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),NaN,NaN,Civilians (Sudan),Government of Sudan (2019-),NaN,NaN,NaN,729,Northern Africa,Sudan,South Darfur,Nyala Janoub,NaN,Nyala,12.0556,24.8906,2,Darfur 24,Subnational,"Looting: On 15 April 2023, an unidentified arm...",0,NaN,1754958639,2025-08-12 00:30:39+00:00,2023-04-01,2023-04,2023Q2,0,4,2023,0,0,1,0,0,0,0,0,Darfur Communal Militia (Sudan); Civilians (Su...,0,0,1,Darfur
1,SUD19671,2023-04-16,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),NaN,NaN,Civilians (Sudan),Labor Group (Sudan),NaN,NaN,NaN,729,Northern Africa,Sudan,Central Darfur,Zalingi,NaN,Zalingei,12.9075,23.4706,1,Darfur 24; Radio Dabanga,Subnational-National,"Looting: On 16 April 2023, an unidentified arm...",0,NaN,1753971757,2025-07-31 14:22:37+00:00,2023-04-01,2023-04,2023Q2,1,4,2023,0,0,1,0,0,0,0,0,Darfur Communal Militia (Sudan); Civilians (Su...,0,0,1,Darfur
2,SUD19672,2023-04-19,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),NaN,NaN,Civilians (Sudan),MSF: Doctors Without Borders,NaN,NaN,NaN,729,Northern Africa,Sudan,South Darfur,Nyala Janoub,NaN,Nyala,12.0556,24.8906,1,Sudan Nile; Twitter,New media-National,"Looting: On 19 April 2023, armed men (coded as...",0,NaN,1754958639,2025-08-12 00:30:39+00:00,2023-04-01,2023-04,2023Q2,4,4,2023,0,0,1,0,0,0,0,0,Darfur Communal Militia (Sudan); Civilians (Su...,0,0,1,Darfur
3,SUD19674,2023-04-16,2023,1,Strategic developments,Strategic developments,Looting/property destruction,Darfur Communal Militia (Sudan),NaN,NaN,Civilians (Sudan),NaN,NaN,NaN,NaN,729,Northern Africa,Sudan,North Darfur,Kebkabiya,NaN,Kebkabiya,13.6469,24.0991,1,Radio Dabanga,National,"Looting: On 16 April 2023, an unidentified arm...",0,NaN,1753971757,2025-07-31 14:22:37+00:00,2023-04-01,2023-04,2023Q2,1,4,2023,0,0,1,0,0,0,0,0,Darfur Communal Militia (Sudan); Civilians (Su...,0,0,1,Darfur
4,SUD19822,2023-04-25,2023,1,Political violence,Battles,Armed clash,Unidentified Armed Group (Sudan),NaN,NaN,Police Forces of Sudan (2019-) Prison Guards,NaN,NaN,NaN,NaN,729,Northern Africa,Sudan,North Darfur,Al Fasher,NaN,El Fasher,13.6264,25.3559,1,Darfur 24,Subnational,"On 25 April 2023, an armed group in cars and o...",0,NaN,1753971757,2025-07-31 14:22:37+00:00,2023-04-01,2023-04,2023Q2,10,4,2023,1,0,0,0,0,0,0,0,Unidentified Armed Group (Sudan); Police Force...,0,0,0,Darfur


Admin1-month panel preview:


,admin1,macro_region,month,year_month,year,month_number,months_since_war_start,events,fatalities,civilian_targeting_events,violence_against_civilians_events,battles,explosions_remote,strategic_developments,protests_riots,saf_events,rsf_events,distinct_actor_count,unique_locations,unique_admin2,civilian_targeting_events_next_month,target_any_civilian_targeting_next_month,target_high_civilian_targeting_next_month,civilian_actor_events,fatal_event_count,mean_latitude,mean_longitude,events_lag1,fatalities_lag1,civilian_targeting_events_lag1,violence_against_civilians_events_lag1,battles_lag1,explosions_remote_lag1,strategic_developments_lag1,protests_riots_lag1,saf_events_lag1,rsf_events_lag1,civilian_actor_events_lag1,fatal_event_count_lag1,distinct_actor_count_lag1,unique_locations_lag1,unique_admin2_lag1,events_prev3mo,fatalities_prev3mo,civilian_targeting_events_prev3mo,battles_prev3mo,explosions_remote_prev3mo
0,Abyei,Abyei,2023-04-01,2023-04,2023,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6.0,1,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Abyei,Abyei,2023-05-01,2023-05,2023,5,1,12,36,6,6,4,0,2,0,0,0,10,10,1,1.0,1,0,8,8,9.527508,28.551233,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Abyei,Abyei,2023-06-01,2023-06,2023,6,2,6,21,1,1,4,0,1,0,0,0,4,3,1,2.0,1,0,2,5,9.541467,28.515283,12,36,6,6,4,0,2,0,0,0,8,8,10,10,1,12,36,6,4,0
3,Abyei,Abyei,2023-07-01,2023-07,2023,7,3,2,0,2,2,0,0,0,0,0,0,4,1,1,1.0,1,0,2,0,9.589900,28.448600,6,21,1,1,4,0,1,0,0,0,2,5,4,3,1,18,57,7,8,0
4,Abyei,Abyei,2023-08-01,2023-08,2023,8,4,7,11,1,1,5,0,0,1,0,0,6,6,1,5.0,1,0,1,6,9.520714,28.536986,2,0,2,2,0,0,0,0,0,0,2,0,4,1,1,20,57,9,8,0


Notebook completed successfully.
